# Tweet Hate Speech Classifier

Author: Julian Steinhauser

## Import & GPU Setup

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

# GPU Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))

## Upload CSV with training data


In [ ]:
from google.colab import files
uploaded = files.upload()

## Preprocessing

In [ ]:
!pip install transformers emoji==0.6.0 --quiet

In [ ]:
from emoji import demojize
from nltk.tokenize import TweetTokenizer
import html

tweet_tokenizer = TweetTokenizer()

def normalizeToken(token):
    lowercased_token = token.lower()
    if token.startswith("@"):
        return "@USER"
    elif lowercased_token.startswith("http") or lowercased_token.startswith("www"):
        return "HTTPURL"
    elif len(token) == 1:
        return demojize(token)
    else:
        if token == "’":
            return "'"
        elif token == "…":
            return "..."
        else:
            return token

def normalizeTweet(tweet):
    tweet = html.unescape(tweet)
    tokens = tweet_tokenizer.tokenize(tweet.replace("’", "'").replace("…", "..."))
    normTweet = " ".join([normalizeToken(token) for token in tokens])

    normTweet = (
        normTweet.replace("cannot ", "can not ")
        .replace("n't ", " n't ")
        .replace("n 't ", " n't ")
        .replace("ca n't", "can't")
        .replace("ai n't", "ain't")
    )
    normTweet = (
        normTweet.replace("'m ", " 'm ")
        .replace("'re ", " 're ")
        .replace("'s ", " 's ")
        .replace("'ll ", " 'll ")
        .replace("'d ", " 'd ")
        .replace("'ve ", " 've ")
    )
    normTweet = (
        normTweet.replace(" p . m .", "  p.m.")
        .replace(" p . m ", " p.m ")
        .replace(" a . m .", " a.m.")
        .replace(" a . m ", " a.m ")
    )

    return " ".join(normTweet.split())

## Load CSV and prepare data

In [ ]:
# load train.csv
df = pd.read_csv("train.csv", header=None, names=['text', 'label']) # you can use your own filename here (instead of train.csv)

# delete header
df = df.drop(df.index[[0,0]])

# normalize
df['text'] = df['text'].apply(normalizeTweet)

# extract text and labels
sentences = df['text'].values.astype(str)
labels = df['label'].values.astype(np.int64)

print(df.head(10))


## Load BERTweet-Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

## Tokenization with the BERTweet-Tokenizer

In [ ]:
MAX_LEN = 128

tokenized_texts = tokenizer(
    list(sentences),
    padding='max_length',
    truncation=True,
    max_length=MAX_LEN,
    return_tensors='np',
    return_attention_mask=True
)

input_ids = tokenized_texts['input_ids']
attention_masks = tokenized_texts['attention_mask']

## Data split

In [ ]:
train_inputs, validation_inputs, train_labels, validation_labels = train_test_split(
    input_ids, labels, test_size=0.1, random_state=42
)

train_masks, validation_masks = train_test_split(
    attention_masks, test_size=0.1, random_state=42
)


## Create TorchTensors

In [ ]:
train_inputs = torch.tensor(train_inputs, dtype=torch.long)
validation_inputs = torch.tensor(validation_inputs, dtype=torch.long)
train_labels = torch.tensor(train_labels, dtype=torch.long)
validation_labels = torch.tensor(validation_labels, dtype=torch.long)
train_masks = torch.tensor(train_masks, dtype=torch.float)
validation_masks = torch.tensor(validation_masks, dtype=torch.float)

## Create TensorDataset

In [ ]:
train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
validation_dataset = TensorDataset(validation_inputs, validation_masks, validation_labels)

## Initialize DataLoader

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

batch_size = 16

train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size
)

validation_dataloader = DataLoader(
    validation_dataset,
    sampler=SequentialSampler(validation_dataset),
    batch_size=batch_size
)


## Load BERTweet-base

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    num_labels=3
)

model.to(device)


## Set hyperparameters

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# Scheduler
epochs = 3
total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)


## Define loss function

In [ ]:
from torch.nn import CrossEntropyLoss

loss_fn = CrossEntropyLoss()

## Training loop

In [ ]:
for epoch in range(epochs):
    print(f"Epoch {epoch+1} / {epochs}")

    # Training
    model.train()

    total_loss = 0
    for step, batch in enumerate(train_dataloader):
        b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

        model.zero_grad()
        outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs.loss
        logits = outputs.logits

        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        # Logging
        if step % 50 == 0 and step != 0:
            print(f"Step {step}, Avg. Loss: {total_loss/(step+1):.4f}")

    avg_train_loss = total_loss / len(train_dataloader)
    print("Avg. training loss:", avg_train_loss)

    # Evaluation
    model.eval()

    eval_loss = 0
    eval_accuracy = 0
    nb_eval_steps = 0

    for batch in validation_dataloader:
        b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

        with torch.no_grad():
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            logits = outputs.logits

        eval_loss += loss.item()
        preds = torch.argmax(logits, dim=1).flatten()
        eval_accuracy += (preds == b_labels).cpu().numpy().mean()
        nb_eval_steps += 1

    # Logging
    print("Validation loss:", eval_loss/nb_eval_steps)
    # Use only if classes are balanced!
    print("Validation accuracy:", eval_accuracy/nb_eval_steps)

## Upload CSV with test data

In [ ]:
uploaded = files.upload()

new_df = pd.read_csv("test_no_label.csv") # you can use your filename instead

new_df.head()

## Preprocessing the test data

In [ ]:
new_df['text'] = new_df['text'].apply(normalizeTweet)

test_sentences = new_df['text'].astype(str).tolist()

for i in range(10):
  print(test_sentences[i])

## Initialize TensorDataset and DataLoader for the test data

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, SequentialSampler

test_encodings = tokenizer(
    test_sentences,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

test_dataset = TensorDataset(
    test_encodings['input_ids'],
    test_encodings['attention_mask']
)

batch_size = 16
test_dataloader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=batch_size)

## Collect predictions

In [ ]:
# Evaluation
model.eval()
predictions = []

for batch in test_dataloader:
    b_input_ids, b_attention_mask = [t.to(device) for t in batch]

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(preds)

## Prepare output and save predictions to CSV

In [ ]:
output_df = pd.DataFrame({
    'id': range(0, len(predictions)),
    'label': predictions
})

output_df.to_csv("test_with_label.csv", index=False) # you can use your filename instead

## Download

In [ ]:
from google.colab import files
files.download("test_with_label.csv") # you can use your filename instead